# 基于 ModelArts 与 MindSpore 的 StyleGAN2 1024×1024 人脸生成与微调案例

**平台：** 华为云 ModelArts Notebook  
**框架：** MindSpore 2.7.2  
**硬件：** Ascend 910B4 NPU  
**模型：** StyleGAN2  
**数据集：** FFHQ 1024×1024 子集  

## 实验目标

本案例在华为云 ModelArts 平台上完成 StyleGAN2 高分辨率人脸生成实验。实验首先加载 FFHQ 预训练模型生成 1024×1024 人脸图像，随后使用 FFHQ 子集继续训练，并在相同随机种子条件下比较微调前后的生成结果。

本案例重点展示：

1. ModelArts、MindSpore 与 Ascend NPU 实验环境；
2. FFHQ 子集的数据统计与样例可视化；
3. 预训练模型推理与 FFHQ 子集微调流程；
4. `truncation_psi=0.7` 与 `truncation_psi=1.0` 条件下的结果对比；
5. 微调前后差异热力图与辅助定量分析；
6. 训练过程、模型权重和输出文件的可视化展示。

> **运行说明**
>
> 本 Notebook 已改为**项目内相对路径**，不再依赖某台机器上的 `/home/...` 或 `/mnt/workspace/...` 绝对目录。  
> StyleGAN2 的训练与推理也全部改为 Notebook 代码单元直接调用，无需再复制命令到 Terminal。
>
> 推荐从上到下依次运行。首次运行会下载数据与预训练 checkpoint，并执行训练/推理；后续再次运行时，如果对应结果已经存在，会自动跳过耗时步骤。若希望强制重新训练或重新推理，可在第 6 节将 `FORCE_RETRAIN` / `FORCE_REINFER` 改为 `True`。


# 1. 实验环境与设备检查

In [ ]:
# 本单元读取当前 Notebook kernel 的环境信息，不主动初始化 Ascend。
# 后续训练和推理会使用当前 kernel 对应的 Python 解释器直接启动 StyleGAN2 脚本。

import sys
import platform
import subprocess
from importlib.metadata import version, PackageNotFoundError

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Operating system:", platform.platform())

try:
    print("MindSpore version:", version("mindspore"))
except PackageNotFoundError:
    print("MindSpore package was not found in the current kernel.")

print("\n===== Ascend NPU status =====")
try:
    result = subprocess.run(
        ["npu-smi", "info"],
        capture_output=True,
        text=True,
        timeout=30,
        check=False
    )
    print(result.stdout if result.stdout else result.stderr)
except FileNotFoundError:
    print("npu-smi command was not found.")
except subprocess.TimeoutExpired:
    print("npu-smi command timed out.")


### 环境检查分析

`npu-smi info` 用于确认当前环境是否可以看到 Ascend NPU，并观察设备状态。MindSpore 版本通过当前 Notebook kernel 的 Python 环境读取。

后续 StyleGAN2 训练和推理将通过 Python 的 `subprocess` 在 Notebook 中直接启动，因此应确保当前 Notebook 选择的是已经安装 MindSpore、且具备 Ascend 运行环境的 kernel。


# 2. 项目目录与结果文件检查

## 2.1 数据与预训练权重准备

本实验所需的 FFHQ 子集压缩包和 StyleGAN2 checkpoint 位于 AtomGit/GitCode 仓库的 `experiment8` 目录。下面代码会使用 Git LFS 自动下载，并将文件整理到当前 `03_face_guard` 项目内部。

项目目录约定：

```text
03_face_guard/
├── checkpoints/
│   └── stylegan2/
│       ├── network-snapshot-025000-D.ckpt
│       ├── network-snapshot-025000-G.ckpt
│       └── network-snapshot-025000-G_ema.ckpt
├── dataset/
│   ├── ffhq_749.zip
│   └── ffhq_1000/
├── results/
│   └── stylegan2/
├── src/
│   └── stylegan2/
│       ├── train.py
│       └── infer.py
└── 03.02_StyleGAN2_FFHQ.ipynb
```

- 仓库页面：`https://ai.gitcode.com/qq_627548571/face/tree/main/experiment8`
- 开源项目目录：`src/stylegan2/src`；实际训练/推理脚本目录：`src/stylegan2/src`
- 数据：`dataset`
- 预训练权重：`checkpoints/stylegan2`
- 训练、推理和可视化结果：`results/stylegan2`

> 首次运行会下载约 1 GB 以上的大文件。下载过程使用项目内临时缓存，成功整理到 `dataset` / `checkpoints` 后会自动删除临时仓库，避免重复占用磁盘。


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

# ------------------------------------------------------------------
# 1) 自动定位项目根目录：只要求预先保留 src/
# ------------------------------------------------------------------
# 原版会同时要求 dataset/、checkpoints/、results/ 已经存在，
# 这会导致这些运行目录被删除后，Notebook 在 mkdir() 之前就定位失败。
#
# 修改后：
#   - 项目根目录识别只依赖顶层 src/
#   - dataset/、checkpoints/、results/ 缺失时自动创建
#   - src/ 内部的 StyleGAN2 源码仍属于需要持久化保留的项目代码
def find_project_root():
    cwd = Path.cwd()

    def looks_like_project_root(path: Path) -> bool:
        # 顶层只要求 src/ 已存在。
        return (path / "src").is_dir()

    # 正常情况：Notebook 位于项目根目录或其子目录。
    for candidate in [cwd, *cwd.parents]:
        if looks_like_project_root(candidate):
            return Path(os.path.relpath(candidate, cwd))

    # 兼容 Notebook 工作目录位于项目上一级的情况。
    for candidate in cwd.iterdir():
        if candidate.is_dir() and looks_like_project_root(candidate):
            return Path(os.path.relpath(candidate, cwd))

    raise RuntimeError(
        "无法定位项目根目录。此修改版只要求项目根目录中预先存在 `src/`。"
    )


PROJECT_ROOT = find_project_root()

# src/ 是唯一要求预先持久化保留的顶层项目目录。
SRC_ROOT = PROJECT_ROOT / "src"
STYLEGAN2_ROOT = SRC_ROOT / "stylegan2"
CODE_DIR = STYLEGAN2_ROOT / "src"

# 以下目录均属于运行时目录，不要求预先存在。
DATASET_ROOT = PROJECT_ROOT / "dataset"
CHECKPOINT_BASE_ROOT = PROJECT_ROOT / "checkpoints"
RESULTS_BASE_ROOT = PROJECT_ROOT / "results"

# 自动创建顶层运行目录以及本实验使用的子目录。
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_BASE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_BASE_ROOT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_ROOT = CHECKPOINT_BASE_ROOT / "stylegan2"
RESULTS_ROOT = RESULTS_BASE_ROOT / "stylegan2"

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("StyleGAN2 root:", STYLEGAN2_ROOT)
print("StyleGAN2 scripts:", CODE_DIR)
print("Dataset dir  :", DATASET_ROOT)
print("Checkpoint dir:", CHECKPOINT_ROOT)
print("Results dir  :", RESULTS_ROOT)

# ------------------------------------------------------------------
# 2) 最终文件位置：全部位于当前项目内部
# ------------------------------------------------------------------
FFHQ_ZIP = DATASET_ROOT / "ffhq_749.zip"
DATA_EXTRACT_DIR = DATASET_ROOT / "ffhq_1000"

REPO_SNAPSHOT_PREFIX = CHECKPOINT_ROOT / "network-snapshot-025000"
REPO_D_CKPT = CHECKPOINT_ROOT / "network-snapshot-025000-D.ckpt"
REPO_G_CKPT = CHECKPOINT_ROOT / "network-snapshot-025000-G.ckpt"
REPO_G_EMA_CKPT = CHECKPOINT_ROOT / "network-snapshot-025000-G_ema.ckpt"

FINAL_FILES = {
    "experiment8/ffhq_749.zip": FFHQ_ZIP,
    "experiment8/network-snapshot-025000-D.ckpt": REPO_D_CKPT,
    "experiment8/network-snapshot-025000-G.ckpt": REPO_G_CKPT,
    "experiment8/network-snapshot-025000-G_ema.ckpt": REPO_G_EMA_CKPT,
}

REPO_GIT_URLS = [
    "https://gitcode.com/qq_627548571/face.git",
    "https://ai.gitcode.com/qq_627548571/face.git",
]
REPO_BRANCH = "main"

# 临时仓库仅用于 Git LFS 下载；成功后会删除。
DOWNLOAD_CACHE_DIR = PROJECT_ROOT / ".stylegan2_download_cache"


def run_command(cmd, *, env=None, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, env=env, cwd=cwd)


def is_lfs_pointer(path: Path) -> bool:
    """判断文件是否仍只是 Git LFS pointer 文本。"""
    try:
        with path.open("rb") as f:
            head = f.read(200)
        return b"git-lfs.github.com/spec/v1" in head
    except OSError:
        return False


def file_ready(path: Path) -> bool:
    return path.is_file() and not is_lfs_pointer(path)


# ------------------------------------------------------------------
# 3) 仅在最终文件缺失时下载
# ------------------------------------------------------------------
need_download = [
    repo_rel
    for repo_rel, target in FINAL_FILES.items()
    if not file_ready(target)
]

if need_download:
    if shutil.which("git") is None:
        raise RuntimeError("当前环境未找到 git，无法自动下载实验文件。")

    lfs_check = subprocess.run(
        ["git", "lfs", "version"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    if lfs_check.returncode != 0:
        raise RuntimeError(
            "当前环境未安装 Git LFS。请先为当前 Notebook 环境安装 Git LFS，"
            "然后重新运行本单元。"
        )

    # 旧的失败缓存不是完整 Git 仓库时直接清理。
    if DOWNLOAD_CACHE_DIR.exists() and not (DOWNLOAD_CACHE_DIR / ".git").exists():
        shutil.rmtree(DOWNLOAD_CACHE_DIR, ignore_errors=True)

    if not (DOWNLOAD_CACHE_DIR / ".git").exists():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"

        last_error = None
        for repo_url in REPO_GIT_URLS:
            try:
                print(f"尝试克隆下载缓存：{repo_url}")
                run_command(
                    [
                        "git", "clone", "--depth", "1",
                        "--branch", REPO_BRANCH,
                        repo_url, str(DOWNLOAD_CACHE_DIR),
                    ],
                    env=clone_env,
                )
                break
            except subprocess.CalledProcessError as exc:
                last_error = exc
                shutil.rmtree(DOWNLOAD_CACHE_DIR, ignore_errors=True)
        else:
            raise RuntimeError("GitCode 仓库克隆失败，请检查网络或仓库访问权限。") from last_error

    run_command(["git", "-C", str(DOWNLOAD_CACHE_DIR), "lfs", "install", "--local"])

    include_patterns = ",".join(need_download)
    run_command([
        "git", "-C", str(DOWNLOAD_CACHE_DIR), "lfs", "pull",
        f"--include={include_patterns}",
        "--exclude=",
    ])

    # 把真实文件移动到项目语义目录，避免保留一份重复的大仓库缓存。
    for repo_rel in need_download:
        source = DOWNLOAD_CACHE_DIR / repo_rel
        target = FINAL_FILES[repo_rel]

        if not file_ready(source):
            raise RuntimeError(f"Git LFS 下载后仍未得到真实文件：{source}")

        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            target.unlink()

        shutil.move(str(source), str(target))
        print(f"已保存：{target}")

    shutil.rmtree(DOWNLOAD_CACHE_DIR, ignore_errors=True)
else:
    print("数据和预训练权重已经存在，跳过下载。")
    if DOWNLOAD_CACHE_DIR.exists():
        shutil.rmtree(DOWNLOAD_CACHE_DIR, ignore_errors=True)

# 最终再检查一次。
bad_files = [str(path) for path in FINAL_FILES.values() if not file_ready(path)]
if bad_files:
    raise RuntimeError("以下文件缺失或仍是 LFS pointer：\n" + "\n".join(bad_files))

# ------------------------------------------------------------------
# 4) 自动解压 FFHQ 子集
# ------------------------------------------------------------------
existing_images = []
if DATA_EXTRACT_DIR.exists():
    for pattern in ("*.png", "*.jpg", "*.jpeg"):
        existing_images.extend(DATA_EXTRACT_DIR.rglob(pattern))

if not existing_images:
    DATA_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"解压数据集：{FFHQ_ZIP} -> {DATA_EXTRACT_DIR}")
    with zipfile.ZipFile(FFHQ_ZIP, "r") as zf:
        zf.extractall(DATA_EXTRACT_DIR)
else:
    print(f"数据集已解压，跳过：{DATA_EXTRACT_DIR}")

print("\n文件准备完成：")
for path in [FFHQ_ZIP, REPO_D_CKPT, REPO_G_CKPT, REPO_G_EMA_CKPT]:
    print(f"- {path} ({path.stat().st_size / 1024**2:.2f} MB)")


In [ ]:
from pathlib import Path
import pandas as pd

# 这些变量全部由第 2.1 节的 PROJECT_ROOT 派生，不包含机器相关绝对路径。
BASE_DIR = PROJECT_ROOT
DATA_DIR = DATA_EXTRACT_DIR

TRAIN_DIR = RESULTS_ROOT / "train_ffhq749_resume_1024"
BEFORE_07_DIR = RESULTS_ROOT / "infer_before_train"
AFTER_07_DIR = RESULTS_ROOT / "infer_after_ffhq749_train"
BEFORE_10_DIR = RESULTS_ROOT / "infer_before_train_psi1"
AFTER_10_DIR = RESULTS_ROOT / "infer_after_train_psi1"
VIS_DIR = RESULTS_ROOT / "case_visualization"
VIS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PY = CODE_DIR / "train.py"
INFER_PY = CODE_DIR / "infer.py"

checks = [
    ("项目根目录", PROJECT_ROOT),
    ("StyleGAN2 项目目录", STYLEGAN2_ROOT),
    ("StyleGAN2 脚本目录", CODE_DIR),
    ("train.py", TRAIN_PY),
    ("infer.py", INFER_PY),
    ("FFHQ 压缩包", FFHQ_ZIP),
    ("FFHQ 解压目录", DATA_DIR),
    ("预训练 G_ema", REPO_G_EMA_CKPT),
    ("训练输出目录", TRAIN_DIR),
    ("微调前结果 psi=0.7", BEFORE_07_DIR),
    ("微调后结果 psi=0.7", AFTER_07_DIR),
    ("微调前结果 psi=1.0", BEFORE_10_DIR),
    ("微调后结果 psi=1.0", AFTER_10_DIR),
]

path_table = pd.DataFrame([
    {
        "检查项": name,
        "相对路径": str(path),
        "是否存在": path.exists(),
        "PNG 数量": len(list(path.glob("*.png"))) if path.is_dir() else 0,
    }
    for name, path in checks
])

path_table


### 路径检查分析

本 Notebook 的所有实验路径都从项目根目录派生：

- 开源项目目录：`src/stylegan2/src`；实际训练/推理脚本目录：`src/stylegan2/src`
- 数据：`dataset`
- 预训练 checkpoint：`checkpoints/stylegan2`
- 训练、推理和可视化结果：`results/stylegan2`

因此复制整个 `03_face_guard` 文件夹到另一台机器后，不需要修改用户名、工作空间目录或磁盘挂载路径。运行第 2.1 节后，应重点确认 `train.py`、`infer.py`、数据集和预训练 `G_ema` 权重均显示为存在。


# 3. FFHQ 子集统计与质量检查

In [ ]:
from PIL import Image
import pandas as pd

image_files = sorted(
    list(DATA_DIR.rglob("*.png"))
    + list(DATA_DIR.rglob("*.jpg"))
    + list(DATA_DIR.rglob("*.jpeg"))
)

if not image_files:
    raise FileNotFoundError(f"未在数据目录中找到图像：{DATA_DIR}")

records = []
invalid_files = []

for img_path in image_files:
    try:
        with Image.open(img_path) as img:
            records.append({
                "文件名": img_path.name,
                "宽度": img.width,
                "高度": img.height,
                "模式": img.mode,
                "文件大小 MB": round(img_path.stat().st_size / 1024 / 1024, 3),
            })
    except Exception as exc:
        invalid_files.append((img_path.name, str(exc)))

image_info = pd.DataFrame(records)

print("有效图像数量:", len(image_info))
print("损坏或无法读取的图像数量:", len(invalid_files))
print("分辨率分布:")
display(image_info.groupby(["宽度", "高度"]).size().reset_index(name="数量"))

print("前 10 张图像信息:")
display(image_info.head(10))

### 数据统计分析

本实验使用 FFHQ 1024×1024 子集进行继续训练。统计结果应显示约 749 张有效图像，且主要分辨率为 1024×1024。损坏图像数量应为 0。数据检查可以避免截断图片、尺寸异常或格式错误在训练阶段造成中断。

# 4. FFHQ 数据样例可视化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 采用均匀间隔抽样，使展示样例覆盖整个文件列表，而不是只展示前 16 张。
sample_count = min(16, len(image_files))
sample_indices = np.linspace(0, len(image_files) - 1, sample_count, dtype=int)
sample_files = [image_files[i] for i in sample_indices]

fig = plt.figure(figsize=(10, 10))

for i, img_path in enumerate(sample_files):
    img = Image.open(img_path).convert("RGB")
    ax = fig.add_subplot(4, 4, i + 1)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(img_path.name, fontsize=7)

fig.suptitle("FFHQ 1024x1024 Subset Samples", fontsize=16)
fig.tight_layout()

sample_output = VIS_DIR / "ffhq_dataset_samples.png"
fig.savefig(sample_output, dpi=200, bbox_inches="tight")
plt.show()

print("Figure saved to:", sample_output)

### 数据样例分析

FFHQ 子集中的人脸图像分辨率较高，面部区域基本居中，并覆盖不同年龄、发型、肤色、表情和姿态。数据具有一定多样性，适合用于验证 StyleGAN2 高分辨率人脸模型的继续训练流程。

需要注意的是，本实验使用的是 FFHQ 随机子集，而原预训练模型同样基于 FFHQ 数据训练。因此本实验属于**同分布微调**，预期结果主要表现为局部纹理和面部特征调整，而不是明显的跨域风格迁移。

# 5. 模型与训练参数

In [ ]:
training_config = pd.DataFrame([
    ["运行平台", "Notebook + MindSpore/Ascend 环境"],
    ["计算设备", "Ascend NPU"],
    ["深度学习框架", "MindSpore"],
    ["生成模型", "StyleGAN2"],
    ["StyleGAN2 源码", str(CODE_DIR)],
    ["数据集", f"FFHQ 1024×1024 子集，共 {len(image_files)} 张有效图像"],
    ["初始化方式", "从 checkpoints/stylegan2 加载 network-snapshot-025000 的 G / D / G_ema"],
    ["Batch size", 1],
    ["输出分辨率", "1024×1024"],
    ["数据增强", "水平翻转 xflips=True"],
    ["训练输出目录", str(TRAIN_DIR)],
    ["预训练推理权重", str(REPO_G_EMA_CKPT)],
], columns=["项目", "配置"])

training_config


### 参数说明

本实验不从零训练 StyleGAN2，而是先通过第 2.1 节从 AtomGit 仓库下载 `network-snapshot-025000-D.ckpt`、`network-snapshot-025000-G.ckpt` 和 `network-snapshot-025000-G_ema.ckpt`。训练时使用三者共同前缀 `network-snapshot-025000` 作为 `--resume_train`，推理时直接使用 `network-snapshot-025000-G_ema.ckpt`。`G_ema` 是生成器参数的指数滑动平均版本，通常比普通 `G` 权重具有更稳定的推理效果。

文件名中的 `025000` 是该代码实现保存的快照编号；本 Notebook 不把它单独解释为 25,000 kimg。


# 6. 在 Notebook 中完成训练与推理

本节不再要求打开 Terminal。训练和推理统一使用当前 Notebook kernel 的 Python 解释器，通过 `subprocess.run()` 调用 `src/stylegan2/src/train.py` 和 `src/stylegan2/src/infer.py`。

所有传给脚本的文件路径也会根据 `src/stylegan2/src` 自动计算为相对路径，例如 `../../dataset/...`、`../../checkpoints/...` 和 `../../results/...`，因此不会绑定到某台机器的绝对目录。

为了方便反复学习和执行：

- 已存在训练 checkpoint 时默认跳过重复训练；
- 已生成完整 seed 图片时默认跳过重复推理；
- 如确实要重新执行，可把下面的 `FORCE_RETRAIN` 或 `FORCE_REINFER` 设置为 `True`。


In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

# 需要完全重新训练/推理时手动切换为 True。
FORCE_RETRAIN = False
FORCE_REINFER = False


def path_from_code(path: Path) -> str:
    """把项目中的路径转换成相对于 src/stylegan2/src 的路径。"""
    return os.path.relpath(path, CODE_DIR)


def run_stylegan_script(script_name: str, arguments):
    script_path = CODE_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f"找不到脚本：{script_path}")

    command = [sys.executable, script_name, *arguments]

    print("Working directory:", CODE_DIR)
    print("Python:", sys.executable)
    print("Command:")
    print("  " + shlex.join(command))

    subprocess.run(
        command,
        cwd=CODE_DIR,
        check=True,
    )


def latest_g_ema_checkpoint():
    checkpoints = sorted(TRAIN_DIR.glob("network-snapshot-*-G_ema.ckpt"))
    if not checkpoints:
        raise FileNotFoundError(
            f"训练目录中没有找到 G_ema checkpoint：{TRAIN_DIR}"
        )
    return checkpoints[-1]


def inference_complete(directory: Path, expected_count=16):
    return len(list(directory.glob("seed*.png"))) >= expected_count


print("StyleGAN2 runner is ready.")
print("train.py:", TRAIN_PY)
print("infer.py:", INFER_PY)


## 6.1 使用预训练 snapshot 继续训练

下面单元直接运行 `train.py`。`cwd` 被固定为 `src/stylegan2/src`，数据、checkpoint 和输出目录则自动换算成相对于源码目录的路径。


In [ ]:
existing_training_ckpts = sorted(
    TRAIN_DIR.glob("network-snapshot-*-G_ema.ckpt")
) if TRAIN_DIR.exists() else []

if existing_training_ckpts and not FORCE_RETRAIN:
    print("检测到已有训练结果，跳过重复训练：")
    for ckpt in existing_training_ckpts:
        print("-", ckpt)
else:
    TRAIN_DIR.parent.mkdir(parents=True, exist_ok=True)

    train_args = [
        "--device_target=Ascend",
        "--device_id=0",
        f"--data_dir={path_from_code(FFHQ_ZIP)}",
        "--batch_size=1",
        "--start_over=False",
        f"--resume_train={path_from_code(REPO_SNAPSHOT_PREFIX)}",
        "--xflips=True",
        "--img_res=1024",
        "--total_kimg=10",
        "--snap=1",
        f"--out_dir={path_from_code(TRAIN_DIR)}",
    ]

    run_stylegan_script("train.py", train_args)

print("训练阶段完成。")


## 6.2 微调前推理：`truncation_psi=0.7`

使用下载到 `checkpoints/stylegan2` 的预训练 `G_ema` 权重生成 seed 0–15。


In [ ]:
if inference_complete(BEFORE_07_DIR) and not FORCE_REINFER:
    print("psi=0.7 微调前结果已存在，跳过重复推理：", BEFORE_07_DIR)
else:
    BEFORE_07_DIR.mkdir(parents=True, exist_ok=True)

    infer_args = [
        "--device_target=Ascend",
        "--device_id=0",
        "--seed=0-15",
        f"--ckpt={path_from_code(REPO_G_EMA_CKPT)}",
        "--img_res=1024",
        "--truncation_psi=0.7",
        f"--out_dir={path_from_code(BEFORE_07_DIR)}",
    ]

    run_stylegan_script("infer.py", infer_args)

print("生成图片数量:", len(list(BEFORE_07_DIR.glob("seed*.png"))))


## 6.3 微调后推理：`truncation_psi=0.7`

自动寻找训练目录中最新的 `G_ema` checkpoint，再使用相同 seed 0–15 生成结果。


In [ ]:
TRAINED_G_EMA_CKPT = latest_g_ema_checkpoint()
print("微调后推理权重:", TRAINED_G_EMA_CKPT)

if inference_complete(AFTER_07_DIR) and not FORCE_REINFER:
    print("psi=0.7 微调后结果已存在，跳过重复推理：", AFTER_07_DIR)
else:
    AFTER_07_DIR.mkdir(parents=True, exist_ok=True)

    infer_args = [
        "--device_target=Ascend",
        "--device_id=0",
        "--seed=0-15",
        f"--ckpt={path_from_code(TRAINED_G_EMA_CKPT)}",
        "--img_res=1024",
        "--truncation_psi=0.7",
        f"--out_dir={path_from_code(AFTER_07_DIR)}",
    ]

    run_stylegan_script("infer.py", infer_args)

print("生成图片数量:", len(list(AFTER_07_DIR.glob("seed*.png"))))


## 6.4 微调前推理：`truncation_psi=1.0`

In [ ]:
if inference_complete(BEFORE_10_DIR) and not FORCE_REINFER:
    print("psi=1.0 微调前结果已存在，跳过重复推理：", BEFORE_10_DIR)
else:
    BEFORE_10_DIR.mkdir(parents=True, exist_ok=True)

    infer_args = [
        "--device_target=Ascend",
        "--device_id=0",
        "--seed=0-15",
        f"--ckpt={path_from_code(REPO_G_EMA_CKPT)}",
        "--img_res=1024",
        "--truncation_psi=1.0",
        f"--out_dir={path_from_code(BEFORE_10_DIR)}",
    ]

    run_stylegan_script("infer.py", infer_args)

print("生成图片数量:", len(list(BEFORE_10_DIR.glob("seed*.png"))))


## 6.5 微调后推理：`truncation_psi=1.0`

In [ ]:
TRAINED_G_EMA_CKPT = latest_g_ema_checkpoint()

if inference_complete(AFTER_10_DIR) and not FORCE_REINFER:
    print("psi=1.0 微调后结果已存在，跳过重复推理：", AFTER_10_DIR)
else:
    AFTER_10_DIR.mkdir(parents=True, exist_ok=True)

    infer_args = [
        "--device_target=Ascend",
        "--device_id=0",
        "--seed=0-15",
        f"--ckpt={path_from_code(TRAINED_G_EMA_CKPT)}",
        "--img_res=1024",
        "--truncation_psi=1.0",
        f"--out_dir={path_from_code(AFTER_10_DIR)}",
    ]

    run_stylegan_script("infer.py", infer_args)

print("生成图片数量:", len(list(AFTER_10_DIR.glob("seed*.png"))))


# 7. 训练输出文件与模型权重

In [ ]:
if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"训练输出目录不存在：{TRAIN_DIR}")

file_records = []

for path in sorted(TRAIN_DIR.iterdir()):
    if not path.is_file():
        continue

    name = path.name
    if "G_ema" in name:
        description = "指数滑动平均生成器权重，最终推理优先使用"
    elif name.endswith("-G.ckpt"):
        description = "生成器权重"
    elif name.endswith("-D.ckpt"):
        description = "判别器权重"
    elif name == "reals.png":
        description = "真实训练样本可视化"
    elif name == "fakes_init.png":
        description = "继续训练开始时的生成样例"
    elif name.startswith("fakes"):
        description = "继续训练结束后的生成样例"
    else:
        description = "其他训练输出"

    file_records.append({
        "文件名": name,
        "大小 MB": round(path.stat().st_size / 1024 / 1024, 2),
        "说明": description,
    })

training_files = pd.DataFrame(file_records)
training_files

### 输出文件分析

训练输出统一保存在 `results/stylegan2/train_ffhq749_resume_1024`。该目录由训练脚本生成，通常包含 `G`、`D`、`G_ema` checkpoint，以及真实样本和训练过程生成样例。

微调后推理不再硬编码某个 checkpoint 文件名，而是自动选择训练目录中最新的 `network-snapshot-*-G_ema.ckpt`，这样更适合不同运行进度和不同机器环境。


# 8. 微调前后生成结果对比

## 8.1 truncation_psi=0.7 条件下的视觉对比

In [ ]:
def load_seed_image(directory: Path, seed: str):
    path = directory / f"seed{seed}.png"
    if not path.exists():
        raise FileNotFoundError(f"找不到图像：{path}")
    return Image.open(path).convert("RGB")

seeds = ["0000", "0001", "0002", "0003"]

fig = plt.figure(figsize=(14, 7))

for i, seed in enumerate(seeds):
    before_img = load_seed_image(BEFORE_07_DIR, seed)
    after_img = load_seed_image(AFTER_07_DIR, seed)

    ax1 = fig.add_subplot(2, len(seeds), i + 1)
    ax1.imshow(before_img)
    ax1.axis("off")
    ax1.set_title(f"Before fine-tuning\nseed {seed}")

    ax2 = fig.add_subplot(2, len(seeds), i + 1 + len(seeds))
    ax2.imshow(after_img)
    ax2.axis("off")
    ax2.set_title(f"After fine-tuning\nseed {seed}")

fig.suptitle("Before vs After Fine-tuning, truncation psi = 0.7", fontsize=16)
fig.tight_layout()

output_path = VIS_DIR / "before_after_compare_psi07.png"
fig.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()

print("Figure saved to:", output_path)

### psi=0.7 结果分析

`truncation_psi=0.7` 会将潜变量向平均分布收缩，使生成结果更加稳定，但也会限制多样性。在相同随机种子下，微调前后图像的整体构图与主体特征较为接近，局部差异主要体现在表情、五官纹理、发型边缘和背景细节。

由于预训练数据和微调数据均来自 FFHQ，微调不会自然产生强烈的风格迁移，因此“整体相似、局部变化”符合本实验的同分布继续训练设置。

## 8.2 truncation_psi=1.0 条件下的视觉对比

In [ ]:
fig = plt.figure(figsize=(14, 7))

for i, seed in enumerate(seeds):
    before_img = load_seed_image(BEFORE_10_DIR, seed)
    after_img = load_seed_image(AFTER_10_DIR, seed)

    ax1 = fig.add_subplot(2, len(seeds), i + 1)
    ax1.imshow(before_img)
    ax1.axis("off")
    ax1.set_title(f"Before fine-tuning\nseed {seed}")

    ax2 = fig.add_subplot(2, len(seeds), i + 1 + len(seeds))
    ax2.imshow(after_img)
    ax2.axis("off")
    ax2.set_title(f"After fine-tuning\nseed {seed}")

fig.suptitle("Before vs After Fine-tuning, truncation psi = 1.0", fontsize=16)
fig.tight_layout()

output_path = VIS_DIR / "before_after_compare_psi1.png"
fig.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()

print("Figure saved to:", output_path)

### psi=1.0 结果分析

当 `truncation_psi` 设置为 1.0 时，模型保留更完整的潜变量变化范围。与 `psi=0.7` 相比，部分样本在人脸轮廓、表情、发型、皮肤纹理和背景区域上的差异更容易观察。

不过，微调前后的整体身份和构图仍可能保持相似。这并不意味着训练没有生效，而是说明本实验在相同数据域内进行继续训练，主要改变局部生成特征。

## 8.3 差异热力图与辅助定量分析

In [ ]:
import numpy as np

metric_records = []
fig = plt.figure(figsize=(15, 10))

for i, seed in enumerate(seeds):
    before = np.asarray(load_seed_image(BEFORE_10_DIR, seed), dtype=np.float32)
    after = np.asarray(load_seed_image(AFTER_10_DIR, seed), dtype=np.float32)

    if before.shape != after.shape:
        raise ValueError(
            f"seed {seed} 的两张图像尺寸不一致：{before.shape} 与 {after.shape}"
        )

    diff = np.abs(after - before)
    diff_gray = diff.mean(axis=2)

    mean_abs_diff = float(diff.mean())
    changed_ratio = float((diff_gray > 10).mean() * 100)

    metric_records.append({
        "Seed": seed,
        "Mean absolute difference": round(mean_abs_diff, 3),
        "Changed pixels > 10 (%)": round(changed_ratio, 2),
    })

    ax1 = fig.add_subplot(3, len(seeds), i + 1)
    ax1.imshow(before.astype(np.uint8))
    ax1.axis("off")
    ax1.set_title(f"Before\nseed {seed}")

    ax2 = fig.add_subplot(3, len(seeds), i + 1 + len(seeds))
    ax2.imshow(after.astype(np.uint8))
    ax2.axis("off")
    ax2.set_title(f"After\nseed {seed}")

    ax3 = fig.add_subplot(3, len(seeds), i + 1 + 2 * len(seeds))
    ax3.imshow(diff_gray, cmap="hot")
    ax3.axis("off")
    ax3.set_title(f"Difference map\nMAD={mean_abs_diff:.2f}")

fig.suptitle("Pixel Differences Before and After Fine-tuning", fontsize=16)
fig.tight_layout()

heatmap_path = VIS_DIR / "fine_tuning_difference_heatmap.png"
fig.savefig(heatmap_path, dpi=200, bbox_inches="tight")
plt.show()

print("Heatmap saved to:", heatmap_path)

difference_table = pd.DataFrame(metric_records)
difference_table

In [ ]:
# 将每个 seed 的两个差异指标绘制为柱状图，便于比较不同样本的变化程度。
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    difference_table["Seed"],
    difference_table["Mean absolute difference"]
)
axes[0].set_title("Mean Absolute Difference by Seed")
axes[0].set_xlabel("Seed")
axes[0].set_ylabel("MAD")

axes[1].bar(
    difference_table["Seed"],
    difference_table["Changed pixels > 10 (%)"]
)
axes[1].set_title("Changed Pixel Ratio by Seed")
axes[1].set_xlabel("Seed")
axes[1].set_ylabel("Percent")

fig.tight_layout()
metric_chart_path = VIS_DIR / "fine_tuning_difference_metrics.png"
fig.savefig(metric_chart_path, dpi=200, bbox_inches="tight")
plt.show()

print("Metric chart saved to:", metric_chart_path)

### 热力图与指标分析

热力图中颜色较亮的位置表示微调前后像素变化较大的区域。变化通常集中在眼睛、眉毛、嘴部表情、发型边缘、面部纹理和部分背景区域。

- **Mean absolute difference（MAD）**：表示微调前后所有 RGB 像素的平均绝对差异；
- **Changed pixels > 10 (%)**：表示平均通道差异超过 10 个灰度级的像素比例。

这些指标能够证明模型输出发生了可测量变化，但不能直接证明生成质量提高。像素差异也会受到局部位置偏移、纹理改变和背景变化影响，因此本实验将其作为视觉对比的辅助证据，而不是独立的质量评价指标。

# 9. 训练过程可视化

In [ ]:
final_fake_candidates = sorted(
    p for p in TRAIN_DIR.glob("fakes*.png")
    if p.name != "fakes_init.png"
)

if not final_fake_candidates:
    raise FileNotFoundError(f"训练目录中没有找到最终生成样例：{TRAIN_DIR}")

FINAL_FAKES_PATH = final_fake_candidates[-1]

process_images = [
    ("Real training samples", TRAIN_DIR / "reals.png"),
    ("Initial generated samples", TRAIN_DIR / "fakes_init.png"),
    ("Final generated samples", FINAL_FAKES_PATH),
]

fig = plt.figure(figsize=(15, 5))

for i, (title, path) in enumerate(process_images):
    if not path.exists():
        raise FileNotFoundError(f"找不到训练过程图像：{path}")

    img = Image.open(path).convert("RGB")
    ax = fig.add_subplot(1, 3, i + 1)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title)

fig.suptitle("StyleGAN2 Training Process Visualization", fontsize=16)
fig.tight_layout()

training_process_path = VIS_DIR / "training_process.png"
fig.savefig(training_process_path, dpi=200, bbox_inches="tight")
plt.show()

print("Final training sample:", FINAL_FAKES_PATH)
print("Figure saved to:", training_process_path)


### 训练过程分析

- `reals.png` 展示进入训练流程的真实 FFHQ 子集样本；
- `fakes_init.png` 展示加载预训练权重后、继续训练初始阶段的生成结果；
- `fakes025000.png` 展示继续训练完成后的生成结果。

因为模型从 FFHQ 预训练权重开始，初始生成图像已经具有较高质量。微调的目标不是从噪声开始学习人脸结构，而是在已有生成能力上适配当前 FFHQ 子集，因此前后变化主要体现在局部特征和输出分布调整。

# 10. 综合实验结果汇总

In [ ]:
result_summary = pd.DataFrame([
    {
        "实验环节": "预训练模型推理",
        "模型权重": "AtomGit 仓库 network-snapshot-025000-G_ema.ckpt",
        "truncation_psi": "0.7 / 1.0",
        "输出": "1024×1024 人脸图像",
        "主要作用": "建立微调前基线",
    },
    {
        "实验环节": "FFHQ 子集继续训练",
        "模型权重": "AtomGit 仓库 snapshot G / D / G_ema",
        "truncation_psi": "不适用",
        "输出": "G、D、G_ema 快照与训练样例",
        "主要作用": "验证预训练模型微调流程",
    },
    {
        "实验环节": "微调后模型推理",
        "模型权重": "network-snapshot-025000-G_ema.ckpt",
        "truncation_psi": "0.7 / 1.0",
        "输出": "1024×1024 人脸图像",
        "主要作用": "观察微调后的局部特征变化",
    },
    {
        "实验环节": "差异热力图与指标",
        "模型权重": "微调前后 G_ema",
        "truncation_psi": "1.0",
        "输出": "热力图、MAD、变化像素比例",
        "主要作用": "辅助量化微调前后输出差异",
    },
])

result_summary

# 11. 实验结论与局限性

## 实验结论

本案例完成了基于 MindSpore 和 Ascend NPU 的 StyleGAN2 高分辨率人脸生成与继续训练流程：

1. 检查 Notebook kernel、MindSpore 与 Ascend NPU 环境；
2. 通过 Git LFS 自动准备 FFHQ 子集和预训练 checkpoint；
3. 将数据、权重、源码和结果统一组织在 `03_face_guard` 项目目录内部；
4. 在 Notebook Cell 中直接启动 StyleGAN2 训练，无需手工切换 Terminal；
5. 使用相同随机种子分别完成微调前后的 `psi=0.7` 和 `psi=1.0` 推理；
6. 使用差异热力图、平均绝对差异和变化像素比例辅助分析输出变化；
7. 可视化训练样本、训练初始结果和训练结束结果。

路径全部由项目根目录动态派生，没有写死用户名、工作空间挂载点或机器绝对目录。因此复制整个 `03_face_guard` 文件夹到另一套兼容的 MindSpore/Ascend 环境后，可以继续按 Notebook 顺序运行。

## 局限性

1. 本实验使用的 FFHQ 子集规模有限；
2. 微调数据与预训练数据同源，视觉变化通常不如跨域数据微调明显；
3. MAD 与变化像素比例只能衡量输出差异，不能直接代表生成质量；
4. 尚未加入 FID、KID、LPIPS 或人工评价等更完整指标；
5. 实际训练速度和可运行性仍依赖使用者的 MindSpore、Ascend 驱动与算子环境是否配置正确。

## 后续改进方向

后续可增加 FID、LPIPS 等评价指标，并继续探索风格混合、噪声控制和潜空间编辑等可控人脸生成实验。

## 伦理说明

本实验生成的是虚构人脸，仅用于课程教学和生成模型研究。实验结果不应用于冒充真实人物、身份欺诈、绕过人脸识别或未经授权的真实人物深度伪造。
